## Classical After-Generation Watermarking Methods

To benchmark Tree-Ring watermarking—which embeds signals in the **initial noise prior to generation**—we include three widely studied **after-generation (post-hoc)** classical watermarking baselines. These methods operate directly on the **final generated image** in either the frequency or transform domains. All three are **training-free**, deterministic, and use analytical verification (no learned detectors).

---

### 1. DFT Single-Frequency Watermark

**Concept:**  
A simple but effective frequency-domain watermark that boosts the magnitude of a specific DFT (Discrete Fourier Transform) coefficient in the image’s frequency spectrum.

**Embedding:**  
Modify a known (u₀, v₀) frequency bin in the grayscale FFT:

$$
F[u_0, v_0] \leftarrow F[u_0, v_0] + \alpha \cdot \text{meanMag}
$$

**Verification:**  
Compute FFT of the test image and check the magnitude at (u₀, v₀).  
A higher magnitude indicates the presence of the watermark.

---

### 2. DWT–DCT Watermark

**Concept:**  
A multi-resolution watermark that embeds a pattern inside the **LL subband** after a DWT (Discrete Wavelet Transform) and DCT-like frequency decomposition. This exploits the stability of mid-frequency components.

**Embedding:**  
1. Apply a 1-level DWT → LL, LH, HL, HH  
2. Apply DCT-like transform (FFT2) on LL  
3. Add a watermark pattern `W` into the LL frequency patch  
4. Inverse FFT2 → inverse DWT

In 1-level 2D DWT, the image is decomposed like this:

```
+---------+---------+
|   LL    |   LH    |
| (coarse)| (edges) |
+---------+---------+
|   HL    |   HH    |
| (edges) | (fine)  |
+---------+---------+
```
**Verification:**  
Repeat DWT → DCT, extract the same patch, compute the **correlation** with the original watermark pattern.

---

### 3. DWT–DCT–SVD Watermark

**Concept:**  
An enhanced hybrid method embedding information in the **singular values** of the DCT-transformed LL subband. Singular values are stable under noise and compression, improving robustness.

**Embedding:**  
1. Apply DWT → LL  
2. Apply DCT-like transform (FFT2) on LL  
3. Perform SVD:  
$$
D = U S V^T
$$  
4. Add watermark signal to top-K singular values:  
$$
S'_i = S_i + \alpha W_i
$$  
5. Reconstruct: $U S' V^T$ → inverse DWT

**Verification:**  
Extract LL, apply DCT and SVD, compute **correlation** between extracted singular values and the known watermark vector.

---

## Summary Comparison Table

### Table 1 — High-Level Characteristics

| Method | Domain | Where It Embeds | Signal Type | Typical Robustness | Blind? |
|--------|--------|------------------|--------------|----------------------|--------|
| **DFT Single-Frequency** | Fourier | Specific (u, v) bin | Scalar magnitude boost | Weak–moderate (fragile to geometric transforms) | ✔ Fully blind |
| **DWT–DCT** | Wavelet + frequency | LL subband DCT patch | 2D pattern | Moderate (robust to slight JPEG/blur) | Semi-blind (requires W) |
| **DWT–DCT–SVD** | Wavelet + frequency + SVD | Singular values of LL | 1D vector | Stronger (stable under noise/JPEG) | Semi-blind (requires W) |

---

### Table 2 — Embedding & Verification Procedures

| Method | Embedding Procedure | Verification Procedure |
|--------|----------------------|------------------------|
| **DFT Single-Frequency** | FFT2 → boost magnitude at (u₀, v₀) → iFFT | FFT2 → read magnitude at (u₀, v₀) → threshold |
| **DWT–DCT** | DWT → 2D DCT/FFT2 → patch + W → inverse | DWT → DCT/FFT2 → extract patch → correlation(W) |
| **DWT–DCT–SVD** | DWT → DCT → SVD → modify S (top-K) → inverse | DWT → DCT → SVD → correlate S with W |

---

### Table 3 — Key Advantages and Limitations

| Method | Advantages | Limitations |
|--------|------------|-------------|
| **DFT Single-Frequency** | Simple, fast, blind detection | Easily destroyed by rotation/crop; weak hiding capacity |
| **DWT–DCT** | Good robustness vs invisibility trade-off | Requires known pattern; fragile under strong geometric distortions |
| **DWT–DCT–SVD** | Most robust; stable SVD structure | More complex; semi-blind; susceptible to large geometric warps |


## Abbreviations

| Abbreviation      | Meaning                                                  | Explanation                                                                                                                                |
| ----------------- | -------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ |
| **DWT**           | *Discrete Wavelet Transform*                             | A transform that decomposes an image into multi-resolution subbands (LL, LH, HL, HH). Used to separate coarse structure from fine details. |
| **DCT**           | *Discrete Cosine Transform*                              | Transforms an image block into frequency components. Commonly used in JPEG; ideal for embedding watermarks in mid-frequency coefficients.  |
| **FFT**           | *Fast Fourier Transform*                                 | Efficient algorithm for computing the Discrete Fourier Transform (DFT). Used for the single-frequency watermark baseline.                  |
| **LL**            | *Low–Low subband (approximation)*                        | The coarse, downsampled version of the image obtained after DWT; stable and good for watermark embedding.                                  |
| **SVD**           | *Singular Value Decomposition*                           | Factorizes a matrix into U, S, Vᵀ; watermarking modifies the singular values (S) because they are stable under noise/compression.          |

In [3]:
import os
import glob
import random
import time
from datetime import timedelta
from io import BytesIO

import numpy as np
from PIL import Image, ImageFilter

import pywt
from scipy.fft import dctn, idctn

from sklearn.metrics import roc_auc_score

import pandas as pd
from IPython.display import display

import torch
from torchvision import transforms


# ======================
#        CONFIG
# ======================
NAME = "post_hoc"
EVAL_RESULT_SAVE_DIR = os.path.join("./eval_results/", NAME)

DATASET_NAME = "stablediff_octoweb"
DATA_DIR = f"./verifier_dataset_{DATASET_NAME}/clean"  # folder with clean images ONLY
SEED = 42
IMAGE_SIZE = 512

# Choose watermarking method: "dft_single", "dwt_dct", or "dwt_dct_svd"
WATERMARK_METHOD = "dwt_dct"  # <-- change here to test other methods

# Pattern sizes / params (tuned to be stronger than the first version)
DWT_DCT_PATTERN_SIZE = 32
SVD_K = 64  # number of singular values used for SVD-based watermark

# DWT–DCT strength (was too small before)
ALPHA_DWT_DCT = 0.25  # try 100–400
ALPHA_DWT_DCT_SVD = 5.0  # try 2–10 for SVD

FID_ROOT = "./fid_eval"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ======================
#   Reproducibility
# ======================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ======================
#    Helpers: DCT2
# ======================


def dct2(x):
    """
    2D orthonormal DCT (type II) using scipy.fft.dctn.
    """
    return dctn(x, type=2, norm="ortho")


def idct2(X):
    """
    2D orthonormal inverse DCT (type III) using scipy.fft.idctn.
    """
    return idctn(X, type=2, norm="ortho")  # type=2 with norm="ortho" inverts itself


def pil_to_np_gray(img: Image.Image):
    """Convert PIL to grayscale numpy [0,1]."""
    return np.array(img.convert("L"), dtype=np.float32) / 255.0


def np_gray_to_pil(arr: np.ndarray):
    """Convert grayscale [0,1] array to RGB PIL."""
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    img_gray = Image.fromarray(arr)
    return Image.merge("RGB", [img_gray] * 3)


# ======================
#    Watermark patterns
# ======================


def make_dwt_dct_pattern(size=32):
    rng = np.random.RandomState(SEED)
    w = rng.randn(size, size)
    # normalize to unit variance
    w = (w - w.mean()) / (w.std() + 1e-8)
    return w


WM_PATTERN_DWT_DCT = make_dwt_dct_pattern(DWT_DCT_PATTERN_SIZE)


def make_svd_wm_vector(k=SVD_K):
    rng = np.random.RandomState(SEED + 1)
    w = rng.randn(k)
    w = (w - w.mean()) / (w.std() + 1e-8)
    return w


WM_VECTOR_SVD = make_svd_wm_vector(SVD_K)

# ======================
#   DFT single-frequency
# ======================


def apply_dft_single_frequency(
    img: Image.Image, strength=10.0, u0_frac=0.25, v0_frac=0.25
):
    """
    Embed watermark by boosting one mid-frequency DFT coefficient in grayscale channel.
    """
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32)

    F = np.fft.fft2(arr)
    h, w = F.shape

    u0 = int(h * u0_frac)
    v0 = int(w * v0_frac)

    mean_mag = np.abs(F).mean()
    F[u0, v0] += strength * mean_mag
    F[-u0, -v0] += strength * mean_mag  # symmetric for real-valued input

    arr_w = np.fft.ifft2(F).real
    arr_w = np.clip(arr_w, 0, 255).astype(np.uint8)
    watermarked_gray = Image.fromarray(arr_w)

    return Image.merge("RGB", [watermarked_gray] * 3)


def detect_dft_single_frequency(img: Image.Image, u0_frac=0.25, v0_frac=0.25):
    """
    Detection score: magnitude of chosen frequency bin normalized by
    average magnitude in a local neighborhood.
    """
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32)

    F = np.fft.fft2(arr)
    h, w = F.shape

    u0 = int(h * u0_frac)
    v0 = int(w * v0_frac)

    mag = np.abs(F)

    # small local neighborhood around (u0, v0) (excluding the center)
    u_min = max(0, u0 - 3)
    u_max = min(h, u0 + 4)
    v_min = max(0, v0 - 3)
    v_max = min(w, v0 + 4)
    patch = mag[u_min:u_max, v_min:v_max].copy()
    center = patch[3, 3] if patch.shape[0] > 3 and patch.shape[1] > 3 else mag[u0, v0]

    patch_flat = patch.flatten()
    # exclude exact center from local average if possible
    if patch_flat.size > 1:
        patch_flat = patch_flat[patch_flat != center]
    local_mean = patch_flat.mean() if patch_flat.size > 0 else (mag.mean() + 1e-8)

    score = float(center / (local_mean + 1e-8))
    return score


# ======================
#     DWT–DCT watermark
# ======================


def apply_dwt_dct(img: Image.Image, alpha=ALPHA_DWT_DCT):
    """
    DWT-DCT watermark in LL subband using real DCT2.
    Stronger alpha and proper normalization.
    """
    arr_gray = pil_to_np_gray(img)  # [H,W] in [0,1]

    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = dct2(LL)

    h_ll, w_ll = dct_LL.shape
    hs = min(DWT_DCT_PATTERN_SIZE, h_ll)
    ws = min(DWT_DCT_PATTERN_SIZE, w_ll)

    patch = dct_LL[:hs, :ws]
    wm = WM_PATTERN_DWT_DCT[:hs, :ws]

    # scale embedding relative to local std
    local_std = patch.std() + 1e-8
    dct_LL[:hs, :ws] = patch + alpha * local_std * wm

    LL_w = idct2(dct_LL)
    coeffs2_w = (LL_w, (LH, HL, HH))
    arr_w = pywt.idwt2(coeffs2_w, "haar")

    return np_gray_to_pil(arr_w)


def apply_dwt_dct_color(img: Image.Image, alpha=ALPHA_DWT_DCT):
    """
    DWT-DCT watermark on the R channel only.
    G and B channels are kept as-is, so colour is preserved.
    """
    img = img.convert("RGB")
    r, g, b = img.split()

    # Work on R as grayscale array [0,1]
    r_arr = np.array(r, dtype=np.float32) / 255.0

    # DWT on R channel
    LL, (LH, HL, HH) = pywt.dwt2(r_arr, "haar")
    dct_LL = dct2(LL)

    h_ll, w_ll = dct_LL.shape
    hs = min(DWT_DCT_PATTERN_SIZE, h_ll)
    ws = min(DWT_DCT_PATTERN_SIZE, w_ll)

    wm = WM_PATTERN_DWT_DCT[:hs, :ws]
    dct_LL[:hs, :ws] = dct_LL[:hs, :ws] + alpha * wm  # no local_std

    LL_w = idct2(dct_LL)
    r_w_arr = pywt.idwt2((LL_w, (LH, HL, HH)), "haar")

    # Clip back to [0,255] and to uint8
    r_w = np.clip(r_w_arr * 255.0, 0, 255).astype(np.uint8)
    r_w_pil = Image.fromarray(r_w)

    # Merge back with original G, B
    img_w = Image.merge("RGB", (r_w_pil, g, b))
    return img_w


def detect_dwt_dct(img: Image.Image):
    """
    Detection: normalized correlation between DCT(LL) patch and watermark pattern.
    Uses absolute value of correlation as score.
    """
    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = dct2(LL)

    h_ll, w_ll = dct_LL.shape
    hs = min(DWT_DCT_PATTERN_SIZE, h_ll)
    ws = min(DWT_DCT_PATTERN_SIZE, w_ll)

    extracted = dct_LL[:hs, :ws]
    wm = WM_PATTERN_DWT_DCT[:hs, :ws]

    x = extracted.flatten()
    y = wm.flatten()

    # normalize
    x = (x - x.mean()) / (x.std() + 1e-8)
    y = (y - y.mean()) / (y.std() + 1e-8)

    score = float((x * y).mean())  # ~ correlation
    return abs(score)


# =========================
#   DWT–DCT–SVD watermark
# =========================


def apply_dwt_dct_svd(img: Image.Image, alpha=ALPHA_DWT_DCT_SVD, k=SVD_K):
    """
    Better DWT-DCT-SVD watermark:
    - embed directly into top-k singular values S
    - scale by std(S[:k]) so strength is relative to host
    """
    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = dct2(LL).real

    U, S, Vt = np.linalg.svd(dct_LL, full_matrices=False)
    k = min(k, len(S))

    S_sub = S[:k]
    wm_vec = WM_VECTOR_SVD[:k]

    # scale watermark relative to host singular values
    s_scale = S_sub.std() + 1e-8
    S_new = S.copy()
    S_new[:k] = S_sub + alpha * s_scale * wm_vec

    dct_LL_w = (U * S_new) @ Vt
    LL_w = idct2(dct_LL_w)

    coeffs2_w = (LL_w, (LH, HL, HH))
    arr_w = pywt.idwt2(coeffs2_w, "haar")
    return np_gray_to_pil(arr_w)


def detect_dwt_dct_svd(img: Image.Image, k=SVD_K):
    """
    Detection: cosine similarity between S[:k] and wm vector.
    No mean/std normalization on S, just norm-based.
    """
    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = dct2(LL).real

    U, S, Vt = np.linalg.svd(dct_LL, full_matrices=False)
    k = min(k, len(S))

    S_sub = S[:k].astype(np.float64)
    wm_vec = WM_VECTOR_SVD[:k].astype(np.float64)

    # cosine similarity: (S_sub · wm) / (||S_sub|| * ||wm||)
    num = np.dot(S_sub, wm_vec)
    den = np.linalg.norm(S_sub) * np.linalg.norm(wm_vec) + 1e-8
    score = float(num / den)
    return abs(score)


def apply_dft_single_frequency_color(
    img: Image.Image,
    strength: float = 10.0,
    u0_frac: float = 0.25,
    v0_frac: float = 0.25,
):
    """
    Embed watermark by boosting one mid-frequency DFT coefficient
    in the R channel only. G and B are kept unchanged.
    """
    img = img.convert("RGB")
    r, g, b = img.split()

    # Work on R channel as float32
    arr = np.array(r, dtype=np.float32)

    F = np.fft.fft2(arr)
    h, w = F.shape

    u0 = int(h * u0_frac)
    v0 = int(w * v0_frac)

    mean_mag = np.abs(F).mean()
    F[u0, v0] += strength * mean_mag
    F[-u0, -v0] += strength * mean_mag  # symmetric for real-valued signal

    arr_w = np.fft.ifft2(F).real
    arr_w = np.clip(arr_w, 0, 255).astype(np.uint8)
    r_w = Image.fromarray(arr_w)

    # merge back with original G, B
    return Image.merge("RGB", (r_w, g, b))


# Unified interface
def apply_watermark(img: Image.Image, method: str):
    if method == "dft_single":
        return apply_dft_single_frequency_color(img)
    elif method == "dwt_dct":
        return apply_dwt_dct_color(img)
    elif method == "dwt_dct_svd":
        return apply_dwt_dct_svd(img)
    else:
        raise ValueError(f"Unknown watermark method: {method}")


def detect_watermark(img: Image.Image, method: str):
    if method == "dft_single":
        return detect_dft_single_frequency(img)
    elif method == "dwt_dct":
        return detect_dwt_dct(img)
    elif method == "dwt_dct_svd":
        return detect_dwt_dct_svd(img)
    else:
        raise ValueError(f"Unknown watermark method: {method}")

from attack import (
    make_clean_aug,
    make_jpeg_aug,
    make_msg_app_combo,
    make_down_up_attack,
    make_blur_aug,
    make_random_crop_attack,
    make_occlusion_block,
    make_geom_aug,
)

from attack import make_train_image_augmentations

attack_factories = {
    "clean": lambda: make_clean_aug(IMAGE_SIZE),
    "jpeg_strong": lambda: make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=60),
    "msg_app_combo": lambda: make_msg_app_combo(IMAGE_SIZE),
    "down_up": lambda: make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5),
    "blur": lambda: make_blur_aug(IMAGE_SIZE),
    "random_crop": lambda: make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)),
    "occlusion": lambda: make_occlusion_block(IMAGE_SIZE, box_frac=0.25),
    "geom_warp": lambda: make_geom_aug(IMAGE_SIZE),
    "train_aug_mix": lambda: make_train_image_augmentations(IMAGE_SIZE),
}


# ======================
#       EVAL UTILS
# ======================


def compute_best_accuracy(scores, labels):
    """
    Sweep thresholds over sorted scores to find max accuracy.
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    uniq = np.unique(scores)
    if len(uniq) == 1:
        preds = (scores >= uniq[0]).astype(np.int32)
        return float((preds == labels).mean()), float(uniq[0])

    best_acc = 0.0
    best_thr = uniq[0]
    for thr in uniq:
        preds = (scores >= thr).astype(np.int32)
        acc = (preds == labels).mean()
        if acc > best_acc:
            best_acc = acc
            best_thr = thr
    return float(best_acc), float(best_thr)


def avg(values):
    return sum(values) / len(values) if values else float("nan")


# watermark

WM_IMAGE_DIR = f"./posthoc_dataset_{WATERMARK_METHOD}_{DATASET_NAME}/watermarked"
os.makedirs(WM_IMAGE_DIR, exist_ok=True)
os.makedirs(EVAL_RESULT_SAVE_DIR, exist_ok=True)

# ======================
#         MAIN
# ======================
def main():
    # Collect clean images
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp")
    file_paths = []
    for e in exts:
        file_paths.extend(glob.glob(os.path.join(DATA_DIR, e)))
    file_paths = sorted(file_paths)
    if not file_paths:
        raise RuntimeError(f"No images found in {DATA_DIR}")
    print(f"Found {len(file_paths)} clean images in {DATA_DIR}")
    print(f"Watermark method: {WATERMARK_METHOD}")

    testing_times = 3
    total_start = time.time()

    resize_base = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))

    results = []

    for attack_name, aug_builder in attack_factories.items():
        print("\n" + "#" * 80)
        print(f"# ATTACK: {attack_name}")
        print("#" * 80)

        attack_aug = aug_builder()

        acc_all, auc_all = [], []

        for test_i in range(testing_times):
            iter_start = time.time()
            print("=" * 80)
            print(
                f"[{attack_name}] Test {test_i + 1:3d}/{testing_times:3d}    "
                f"Time elapsed: {str(timedelta(seconds=int(time.time() - total_start)))}"
            )
            print("-" * 80)

            scores = []
            labels = []

            # Build a (clean, watermarked) pair for each image
            for idx, path in enumerate(file_paths):
                img = Image.open(path).convert("RGB")
                img = resize_base(img)

                # clean example (after attack)
                img_clean_attacked = attack_aug(img)
                score_clean = detect_watermark(img_clean_attacked, WATERMARK_METHOD)
                scores.append(score_clean)
                labels.append(0)

                # watermarked example (after attack)
                img_wm = apply_watermark(img, WATERMARK_METHOD)

                # save the image
                if attack_name == "clean" and test_i == 0:
                    # save watermarked image
                    base = os.path.splitext(os.path.basename(path))[0]
                    img_wm.save(os.path.join(WM_IMAGE_DIR, f"{base}_wm.png"))

                img_wm_attacked = attack_aug(img_wm)
                score_wm = detect_watermark(img_wm_attacked, WATERMARK_METHOD)
                scores.append(score_wm)
                labels.append(1)

            labels_np = np.asarray(labels, dtype=np.int32)
            scores_np = np.asarray(scores, dtype=np.float64)

            # AUROC
            try:
                auc = roc_auc_score(labels_np, scores_np)
            except ValueError:
                auc = float("nan")

            # Best accuracy via threshold sweep
            best_acc, best_thr = compute_best_accuracy(scores_np, labels_np)

            acc_all.append(best_acc)
            auc_all.append(auc)

            iter_time = time.time() - iter_start
            col1_w = 14
            col_w = 18
            print(f"{'Metric':<{col1_w}} {'Value':>{col_w}}")
            print("-" * (col1_w + col_w + 3))
            print(f"{'Best_Acc':<{col1_w}} {best_acc:>{col_w}.4f}")
            print(f"{'Best_Thr':<{col1_w}} {best_thr:>{col_w}.4f}")
            print(f"{'AUROC':<{col1_w}}   {auc:>{col_w}.4f}")
            print("-" * (col1_w + col_w + 3))
            print(f"Iter time: {iter_time:.1f}s")
            print("=" * 80)

        # store averages for this attack
        results.append(
            {
                "attack": attack_name,
                "accuracy": avg(acc_all),
                "auc": avg(auc_all),
            }
        )

    print("\n" + "=" * 80)
    print("All evaluations complete.")
    print("=" * 80)
    df_results = pd.DataFrame(results)
    display(df_results)
    # save results to CSV
    csv_path = os.path.join(EVAL_RESULT_SAVE_DIR, "attack_eval_summary.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"Saved summary results to {csv_path}")

if __name__ == "__main__":
    main()

Found 500 clean images in ./verifier_dataset_stablediff_octoweb/clean
Watermark method: dwt_dct

################################################################################
# ATTACK: clean
################################################################################
[clean] Test   1/  3    Time elapsed: 0:00:00
--------------------------------------------------------------------------------


KeyboardInterrupt: 